# Iteration 2 — Melancholic Poet QLoRA Training

Trains both adapters (persona-only and mixed) with:
- Filtered data: only `high` + `medium` persona strength (177 examples, dropping 85 `low`)
- Lower learning rate: 5e-5
- More epochs: 8 (persona-only), 6 (mixed)

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf /content/PET_QLORA_POET
!git clone https://github.com/thejesusyung/melancholic-poet-qlora.git /content/PET_QLORA_POET

In [ ]:
%cd /content/PET_QLORA_POET/melancholic_poet_qlora
!pip install -r requirements.txt -q

import os, sys
os.environ["PYTHONPATH"] = "/content/PET_QLORA_POET/melancholic_poet_qlora/src"
sys.path.insert(0, "/content/PET_QLORA_POET/melancholic_poet_qlora/src")

## 2. Prepare filtered training data

In [ ]:
!python scripts/prepare_root_data.py \
  --source_dir /content/PET_QLORA_POET/data \
  --min_persona_strength medium

## 3. Train persona-only adapter

In [ ]:
!python train.py --config configs/persona_only_qwen25_15b.yaml

## 4. Train mixed adapter

In [ ]:
!python train.py --config configs/mixed_qwen25_15b.yaml

## 5. Upload adapters to S3

Replace the old adapters so the EC2 Gradio app picks them up on next restart.

In [ ]:
S3_BUCKET = "melancholic-poet-qlora-901059153423-us-east-2-an"

!pip install awscli -q
print("Paste your AWS credentials below (or skip if using Colab secrets):")

In [ ]:
import os
# Uncomment and fill in, or set via Colab secrets
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

In [ ]:
!aws s3 sync outputs/persona_only_qwen25_15b/adapter/ s3://{S3_BUCKET}/adapters/persona_only/ --delete
!aws s3 sync outputs/mixed_qwen25_15b/adapter/ s3://{S3_BUCKET}/adapters/mixed/ --delete
print("Done. Restart the EC2 Gradio app to load the new adapters.")